# Cumulative MTD Export Pace — Kpler Flows prototype

Pulls **daily export flows** for a location + calendar month from the Kpler **Flows** endpoint,
builds the cumulative month-to-date (MTD) curve, and overlays:

- **Current month** (partial, dotted) — truncated at today
- **Last month** and **Last year** (same as your Excel chart)
- **N-year seasonal band**: median + interquartile (25–75%) + min–max range, computed
  per *day-of-month* across the prior N years of the *same calendar month*

### Design decisions worth knowing (these are easy to get wrong)
1. **Flows is the right endpoint.** It returns aggregated import/export flows for a zone or
   installation at daily granularity — exactly the time series this chart needs. (`Trades` gives
   you cargo-level rows; you'd have to aggregate yourself. Don't.)
2. **Pull volume, not rate, then cumulate.** We request the unit **`KB`** (thousand barrels =
   volume *that day*) and `cumsum()` it, so the y-axis reads "thousand barrels shipped MTD".
   Your Excel chart is labelled *kbd* but is really this cumulative curve — cumulating a *rate*
   (kbd) isn't a clean physical unit, so KB is the honest choice. Switch the unit in config if
   your desk insists on the kbd convention; the shape is identical.
3. **Forecast handling.** Flows defaults to `with_forecast=True` (Kpler's model fills in expected
   loadings). For a clean MTD *pace* comparison you usually want **realized only** for history and
   a deliberate choice for the live month. Both flags are exposed in config — see notes at the bottom.
4. **Day-of-month alignment.** We index each month by day-of-month (1..31) so years line up. For the
   *same* calendar month across years this is clean (only Feb leap-day varies; handled nan-aware).
   "Last month" is a *different* calendar month, so it's a rough pace overlay — fine for context,
   not a like-for-like.

**Run it now with `USE_MOCK = True` to see the chart, then flip to `False` and add credentials.**

In [1]:
import calendar
from datetime import date
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path

# ----------------------------- CONFIG -----------------------------
USE_MOCK     = False          # True = synthetic data, runs with zero setup. False = real Kpler.

LOCATION     = 'Russia' #["Yanbu", "Fujairah", "Oman"]  # str or list[str] — zone / installation names
PRODUCT      = "Gasoil/Diesel"                     # str or list[str] — Kpler product name(s)
MONTH        = 10                  # calendar month to analyse (1-12)
TARGET_YEAR  = 2026               # the 'current' year for this view
N_HISTORY    = 5                  # band over the prior N same-calendar-months
UNIT         = "kb"               # 'kb' (volume/day -> clean cumsum) or 'kbd'
AS_OF_DAY    = None                 # last day to draw for the 'current month' line.
                                  #   None -> use today if it's the live month, else full month.

# Kpler API params (only used when USE_MOCK = False)
WITH_FORECAST_LIVE = False    # include model forecast for the current/partial month
WITH_FORECAST_HIST = False   # history: realized picture is usually what you want
ONLY_REALIZED_HIST = True
WITH_INTRA_COUNTRY = True    # count intra-country trades as exports? depends on the location

# Credentials (env vars recommended; never hard-code in a shared notebook)
import os

load_dotenv(dotenv_path=Path('.env'))


KPLER_EMAIL    = os.getenv("KPLER_EMAIL")
KPLER_PASSWORD = os.getenv("KPLER_PASSWORD")


def as_filter_list(value):
    """Normalize LOCATION / PRODUCT: one name or a list for Flows API filters."""
    if not value:
        return []
    if isinstance(value, str):
        return [value]
    return list(value)


def filter_label(value, *, sep=", "):
    """Readable label for titles (single name unchanged; lists joined)."""
    items = as_filter_list(value)
    return items[0] if len(items) == 1 else sep.join(items)


LOCATIONS = as_filter_list(LOCATION)
PRODUCTS = as_filter_list(PRODUCT)
LOCATION_LABEL = filter_label(LOCATION)
PRODUCT_LABEL = filter_label(PRODUCT)
# ------------------------------------------------------------------

In [2]:
KPLER_EMAIL

'luiscarlos.gaitan@jainglobal.com'

## 1. Data layer — real Kpler Flows call, with a mock fallback

In [3]:
_flows_client = None
_flows_bulk = {}  # cache key -> raw Flows DataFrame (one API call per policy + filters)


def _get_flows_client():
    """Lazily build one Flows client (reused across calls)."""
    global _flows_client
    if _flows_client is None:
        from kpler.sdk import Platform
        from kpler.sdk.configuration import Configuration
        from kpler.sdk.resources.flows import Flows
        config = Configuration(Platform.Liquids, KPLER_EMAIL, KPLER_PASSWORD)
        _flows_client = Flows(config)
    return _flows_client


def _flows_bulk_start_date():
    """Earliest calendar day needed for history band, overlays, and prior-year month."""
    return date(TARGET_YEAR - N_HISTORY - 1, 1, 1)


def _flows_bulk_cache_key(live):
    """Cache key so bulk pull refreshes when zones, products, or API flags change."""
    return (
        live,
        tuple(LOCATIONS),
        tuple(PRODUCTS),
        UNIT,
        WITH_FORECAST_LIVE if live else WITH_FORECAST_HIST,
        False if live else ONLY_REALIZED_HIST,
        WITH_INTRA_COUNTRY,
    )


def _fetch_flows_bulk(*, live=False):
    """One Flows request for the full span; sliced per (year, month) downstream."""
    key = _flows_bulk_cache_key(live)
    if key in _flows_bulk:
        return _flows_bulk[key]
    from kpler.sdk import (FlowsDirection, FlowsSplit, FlowsPeriod,
                           FlowsMeasurementUnit)
    unit_enum = {"kb": FlowsMeasurementUnit.KB,
                 "kbd": FlowsMeasurementUnit.KBD}[UNIT]
    df = _get_flows_client().get(
        flow_direction=[FlowsDirection.Export],
        split=[FlowsSplit.Total],
        granularity=[FlowsPeriod.Daily],
        from_zones=LOCATIONS,
        products=PRODUCTS,
        unit=[unit_enum],
        start_date=_flows_bulk_start_date(),
        end_date=date.today(),
        with_forecast=WITH_FORECAST_LIVE if live else WITH_FORECAST_HIST,
        only_realized=False if live else ONLY_REALIZED_HIST,
        with_intra_country=WITH_INTRA_COUNTRY,
    )
    _flows_bulk[key] = df
    return df


def _flows_df_to_daily(df, year, month):
    """Extract one calendar month from a Flows bulk frame (Date + Total columns)."""
    dt = pd.to_datetime(df["Date"])
    if "Total" in df.columns:
        values = pd.to_numeric(df["Total"], errors="coerce")
    else:
        # split != Total: sum numeric flow columns (exclude Period End Date, etc.)
        values = df.select_dtypes("number").sum(axis=1)
    s = pd.Series(values.to_numpy(), index=dt, name="export")
    return s[(s.index.month == month) & (s.index.year == year)].sort_index()


def _kpler_export_daily(year, month, *, live=False):
    """Daily export series for one month, sliced from the bulk Flows pull."""
    df = _fetch_flows_bulk(live=live)
    return _flows_df_to_daily(df, year, month)


def _mock_export_daily(year, month, *, live=False, base=600.0, seed=0):
    # Tie mock noise to the active zone/product filters.
    filter_seed = hash((tuple(LOCATIONS), tuple(PRODUCTS))) % (2**31)
    rng = np.random.default_rng(seed + filter_seed + year * 100 + month)
    n = calendar.monthrange(year, month)[1]
    trend = 1.0 + 0.03 * (year - 2020)
    seasonal = 1.0 + 0.12 * np.sin((month - 1) / 12 * 2 * np.pi)
    mean_daily = base * trend * seasonal
    cargo = 380.0
    daily = rng.poisson(1.6, n) * cargo * rng.uniform(0.6, 1.4, n)
    daily = daily * (mean_daily / max(daily.mean(), 1e-9))
    idx = pd.date_range(date(year, month, 1), periods=n, freq="D")
    return pd.Series(daily, index=idx, name="export")


_cache = {}
def get_export_daily(year, month, *, live=False):
    """Cached daily export series for one (year, month)."""
    key = (year, month, live, tuple(LOCATIONS), tuple(PRODUCTS))
    if key not in _cache:
        fn = _mock_export_daily if USE_MOCK else _kpler_export_daily
        _cache[key] = fn(year, month, live=live)
    return _cache[key]

## 2. Transform — cumulative by day-of-month, and the history matrix

In [4]:
def to_dom_cumulative(daily, *, as_of=None):
    """Reindex to day-of-month (1..N) and cumulative-sum. as_of truncates the live month."""
    s = daily.copy()
    if as_of is not None:
        s = s[s.index.date <= as_of]
    cum = s.cumsum()
    cum.index = cum.index.day
    return cum


def prev_month(year, month):
    return (year - 1, 12) if month == 1 else (year, month - 1)


def build_history_matrix(month, years):
    """day_of_month (rows) x year (cols) cumulative-export matrix."""
    return pd.DataFrame(
        {y: to_dom_cumulative(get_export_daily(y, month)) for y in years}
    ).sort_index()


def seasonal_band(hist):
    out = pd.DataFrame(index=hist.index)
    out["median"] = hist.median(axis=1)
    out["p25"]    = hist.quantile(0.25, axis=1)
    out["p75"]    = hist.quantile(0.75, axis=1)
    out["min"]    = hist.min(axis=1)
    out["max"]    = hist.max(axis=1)
    return out

## 3. Assemble the series for this view

In [5]:
# resolve 'as of' for the current-month line
today = date.today()
is_live_month = (TARGET_YEAR == today.year and MONTH == today.month)
if AS_OF_DAY is not None:
    as_of = date(TARGET_YEAR, MONTH, AS_OF_DAY)
elif is_live_month:
    as_of = today
else:
    as_of = None   # full month

hist_years = list(range(TARGET_YEAR - N_HISTORY, TARGET_YEAR))   # prior N same-months
hist = build_history_matrix(MONTH, hist_years)
band = seasonal_band(hist)

current  = to_dom_cumulative(get_export_daily(TARGET_YEAR, MONTH, live=True), as_of=as_of)
last_yr  = to_dom_cumulative(get_export_daily(TARGET_YEAR - 1, MONTH))
py, pm   = prev_month(TARGET_YEAR, MONTH)
last_mo  = to_dom_cumulative(get_export_daily(py, pm))

band.tail()

,median,p25,p75,min,max
Date,,,,,
27,21025.19,17783.10,22116.43,17712.44,23593.02
28,22202.46,18204.15,23406.28,17783.41,24331.89
29,23314.05,19684.81,24862.10,18986.52,25789.79
30,23730.28,20249.44,25565.50,19373.09,26435.36
31,24442.20,21147.79,26077.72,19896.59,27094.79


## 4. Plot (plotly — this `go.Figure` drops straight into a Dash `dcc.Graph`)

In [6]:
import plotly.graph_objects as go

month_name = calendar.month_name[MONTH]
unit_label = "kb" if UNIT == "kb" else "kbd (cumulated)"
x = band.index

fig = go.Figure()
# min-max range band
fig.add_trace(go.Scatter(x=x, y=band["max"], line=dict(width=0), hoverinfo="skip",
                         showlegend=False))
fig.add_trace(go.Scatter(x=x, y=band["min"], fill="tonexty", line=dict(width=0),
                         fillcolor="rgba(158,202,225,0.30)",
                         name=f"{N_HISTORY}y range (min–max)", hoverinfo="skip"))
# interquartile band
fig.add_trace(go.Scatter(x=x, y=band["p75"], line=dict(width=0), hoverinfo="skip",
                         showlegend=False))
fig.add_trace(go.Scatter(x=x, y=band["p25"], fill="tonexty", line=dict(width=0),
                         fillcolor="rgba(107,174,214,0.45)",
                         name=f"{N_HISTORY}y IQR (25–75%)", hoverinfo="skip"))
# median + overlays
fig.add_trace(go.Scatter(x=x, y=band["median"], name=f"{N_HISTORY}y median",
                         line=dict(color="#2171b5", width=2)))
fig.add_trace(go.Scatter(x=last_mo.index, y=last_mo.values, name="Last month",
                         line=dict(color="#9e9ac8", width=2)))
fig.add_trace(go.Scatter(x=last_yr.index, y=last_yr.values, name="Last year",
                         line=dict(color="#74c476", width=2)))
fig.add_trace(go.Scatter(x=current.index, y=current.values, name="Current month",
                         line=dict(color="#e6550d", width=3, dash="dot")))

fig.update_layout(
    title=f"Cumulative {PRODUCT_LABEL} exports — {LOCATION_LABEL} — {month_name} {TARGET_YEAR}"
          + ("  (MOCK DATA)" if USE_MOCK else ""),
    xaxis_title="Day of month", yaxis_title=f"Cumulative {unit_label} (MTD)",
    xaxis=dict(range=[1, 31], dtick=2), template="plotly_white",
    legend=dict(orientation="h", y=-0.2), height=520,
)
fig.show()

## 5. Mapping to the real API & parameters to think about

The live call is `Flows.get(...)`. The parameters that change your numbers:

| Param | What it does | Suggested |
|---|---|---|
| `flow_direction` | `Export` / `Import` / `NetExport` | `Export` |
| `split` | how the total is broken out | `Total` (one curve). Use `DestinationCountries` etc. later for stacked views |
| `granularity` | `Daily` / `Weekly` / `Monthly` | `Daily` for this chart |
| `from_zones` / `products` | export origin zones and product filters | pass one name or a list; Kpler returns one combined `Total` series |
| `from_zones` vs `from_installations` | country/region/port **vs** a specific terminal | zone for country-level; installation for a single terminal (e.g. Ras Tanura) |
| `unit` | `kb` (volume) vs `kbd` (rate) | `kb`, then cumsum |
| `with_forecast` | include Kpler's predicted loadings | `False` for history; decide per-desk for the live month |
| `only_realized` | only completed trades | `True` for clean history |
| `with_intra_country` | count within-country moves | depends on location/product |

**Validate the zone/product names first.** Use the `Zones` and `Products` dimension endpoints
(`kpler.sdk.resources.zones.Zones`, `...products.Products`) to confirm exact strings — Flows is
strict about names. A quick `zones_client.get(...)` lookup avoids silent empty results.

**Response shape.** `Flows.get` returns a `Date` column and `Total` (when `split=Total`), not a
DatetimeIndex — slice on `Date`, not the row index.

**Bulk pull.** Section 1 fetches one date range (at most two calls if live-month `only_realized`
differs from history), then slices each calendar month in pandas. For the live month, realized data
lags ~a day or two behind today.

## 6. Path to a Dash app (when you're ready)

The structure here is deliberately Dash-shaped — lift sections 1–3 into a `data.py` module and
section 4 into a `figure.py`, then the app is thin:

```python
# app.py (sketch)
import dash
from dash import dcc, html, Input, Output
from data import build_view          # wraps sections 1-3 -> (current,last_mo,last_yr,band)
from figure import make_figure        # wraps section 4 -> go.Figure

app = dash.Dash(__name__)
app.layout = html.Div([
    dcc.Dropdown(id="location", options=[...], value="Saudi Arabia"),
    dcc.Dropdown(id="product",  options=[...], value="crude"),
    dcc.Dropdown(id="month",    options=[{"label": m, "value": i} for i, m in ...]),
    dcc.Graph(id="chart"),
])

@app.callback(Output("chart","figure"),
              Input("location","value"), Input("product","value"), Input("month","value"))
def update(location, product, month):
    view = build_view(location, product, month)   # cache this!
    return make_figure(view)
```

Two things that matter once it's an app:
- **Cache the Kpler calls.** History (prior years, realized) is static — cache it on disk
  (`functools.lru_cache`, `joblib.Memory`, or a Snowflake table) so dropdown changes don't
  re-hit the API. Only the live month needs refreshing. This is also where your Snowflake setup
  fits: land facility-day flows into a table and read from there.
- **Populate dropdowns from the `Zones`/`Products` dimension endpoints** so users can't pick
  invalid names.

A notebook is the right call for *now*; promote to Dash once the data layer and the few stats
choices (band percentiles, forecast policy) feel settled.

## 7. Global export pace — country contributors & inflection monitoring

Daily view of **worldwide exports by origin country** (`FlowsSplit.OriginCountries`, no zone
filter). Tracks whether export **pace** (mean daily kbd MTD) is shifting — useful when incremental
supply may be returning and could affect prices.

**Pace metric.** At day *d*, country pace = mean daily kbd from DOM 1…*d*. Compare current month vs:

- **Prior month** (same DOM) — e.g. Jul 1–7 vs Jun 1–7
- **Last year** (same calendar month, same DOM)
- **5y seasonal median** (same calendar month, same DOM) — flags countries above/below historical pattern

**Outputs per product:** summary KPIs, contributor bar charts, heatmaps, text notes, **MTD average pace**
(run-rate), and **cumulative MTD pace** (Section 4 style).

One Kpler API call per product; results cached in-session for re-runs.

In [7]:
# ---------------------- Section 7 config ----------------------
PACE_PRODUCTS = ["Gasoline", "Jet", "Gasoil/Diesel", "FO"]#"Diesel", "Gasoil", "FO"]
PACE_MONTH = None          # None -> current calendar month
PACE_YEAR = None           # None -> current year
PACE_AS_OF_DAY = None      # None -> today when live month, else full month
PACE_TOP_N = 5
PACE_N_HISTORY = 5         # years for seasonal pace band
PACE_WITH_FORECAST = False
PACE_ONLY_REALIZED = True
PACE_WITH_INTRA_COUNTRY = False

_today = date.today()#date(2026,9,30)#
PACE_MONTH = PACE_MONTH or _today.month
PACE_YEAR = PACE_YEAR or _today.year
_is_live = PACE_YEAR == _today.year and PACE_MONTH == _today.month
if PACE_AS_OF_DAY is not None:
    _pace_as_of = date(PACE_YEAR, PACE_MONTH, PACE_AS_OF_DAY)
elif _is_live:
    _pace_as_of = _today
else:
    _pace_as_of = None
_pace_dom = _pace_as_of.day if _pace_as_of else calendar.monthrange(PACE_YEAR, PACE_MONTH)[1]
_py_prior, _pm_prior = prev_month(PACE_YEAR, PACE_MONTH)
_pace_month_name = calendar.month_name[PACE_MONTH]

# Kpler meta columns — not origin countries
_FLOWS_META_COLS = {"Date", "Period End Date", "Total"}

_origin_countries_cache: dict = {}


def _pace_fetch_start() -> date:
    return date(PACE_YEAR - PACE_N_HISTORY - 1, 1, 1)


def _mock_origin_countries_flows(product: str) -> pd.DataFrame:
    """Synthetic wide OriginCountries frame when USE_MOCK=True."""
    start, end = _pace_fetch_start(), date.today()
    idx = pd.date_range(start, end, freq="D")
    seed = hash((product, "origin_countries")) % (2**31)
    rng = np.random.default_rng(seed)
    countries = [
        "United States", "India", "Belgium", "Saudi Arabia", "United Arab Emirates",
        "South Korea", "Oman", "Taiwan", "United Kingdom", "Russian Federation",
        "Singapore", "Nigeria", "Brazil", "Netherlands", "China",
    ]
    bases = rng.uniform(80, 450, len(countries))
    data = {"Date": idx}
    for c, b in zip(countries, bases):
        season = 1.0 + 0.08 * np.sin((idx.month - 1) / 12 * 2 * np.pi)
        noise = rng.uniform(0.4, 1.6, len(idx))
        data[c] = b * season * noise
    return pd.DataFrame(data)


def fetch_origin_countries_flows(product: str) -> pd.DataFrame:
    """One cached Flows pull per product — global exports by origin country, kbd."""
    key = (
        product,
        _pace_fetch_start(),
        date.today(),
        PACE_WITH_FORECAST,
        PACE_ONLY_REALIZED,
        PACE_WITH_INTRA_COUNTRY,
    )
    if key in _origin_countries_cache:
        return _origin_countries_cache[key]
    if USE_MOCK:
        df = _mock_origin_countries_flows(product)
    else:
        from kpler.sdk import (
            FlowsDirection,
            FlowsMeasurementUnit,
            FlowsPeriod,
            FlowsSplit,
        )
        df = _get_flows_client().get(
            flow_direction=[FlowsDirection.Export],
            split=[FlowsSplit.OriginCountries],
            granularity=[FlowsPeriod.Daily],
            products=[product],
            unit=[FlowsMeasurementUnit.KBD],
            start_date=_pace_fetch_start(),
            end_date=date.today(),
            with_forecast=PACE_WITH_FORECAST,
            only_realized=PACE_ONLY_REALIZED,
            with_intra_country=PACE_WITH_INTRA_COUNTRY,
        )
    _origin_countries_cache[key] = df
    return df


def slice_month_by_country(
    df: pd.DataFrame, year: int, month: int, *, as_of: date | None = None
) -> pd.DataFrame:
    """Daily kbd by origin country, indexed by day-of-month."""
    if df.empty or "Date" not in df.columns:
        return pd.DataFrame()
    dt = pd.to_datetime(df["Date"])
    mask = (dt.dt.year == year) & (dt.dt.month == month)
    if as_of is not None:
        mask &= dt.dt.date <= as_of
    sliced = df.loc[mask].copy()
    if sliced.empty:
        return pd.DataFrame()
    cols = [c for c in sliced.columns if c not in _FLOWS_META_COLS]
    for col in cols:
        sliced[col] = pd.to_numeric(sliced[col], errors="coerce").fillna(0)
    out = sliced.set_index(dt.loc[mask].dt.day)[cols]
    return out.sort_index()


def country_mean_pace(daily_dom: pd.DataFrame, thru_dom: int) -> pd.Series:
    """Mean daily kbd from DOM 1..thru_dom for each country."""
    if daily_dom.empty:
        return pd.Series(dtype=float)
    window = daily_dom.loc[daily_dom.index <= thru_dom]
    return window.mean()


def global_mean_pace(daily_dom: pd.DataFrame, thru_dom: int) -> float:
    """Global export pace = sum of country kbd rates, averaged over DOM 1..thru_dom."""
    if daily_dom.empty:
        return float("nan")
    window = daily_dom.loc[daily_dom.index <= thru_dom]
    return float(window.sum(axis=1).mean())

In [8]:
_COUNTRY_SHORT = {
    "United States": "USA",
    "United Arab Emirates": "UAE",
    "Russian Federation": "Russia",
    "United Kingdom": "UK",
    "South Korea": "South Korea",
}


def short_country(name: str) -> str:
    return _COUNTRY_SHORT.get(name, name)


def pace_delta(current: pd.Series, baseline: pd.Series) -> pd.Series:
    """Country-level pace change (kbd); union of all countries seen in either month."""
    all_countries = current.index.union(baseline.index)
    return current.reindex(all_countries, fill_value=0) - baseline.reindex(all_countries, fill_value=0)


def pace_contributors(
    current: pd.Series,
    baseline: pd.Series,
    *,
    top_n: int = 5,
) -> dict:
    delta = pace_delta(current, baseline)
    # Decliners/gainers ranked by magnitude in each direction
    decliners = delta[delta < 0].nsmallest(top_n)
    gainers = delta[delta > 0].nlargest(top_n)
    # Baseline had exports but current month is flat at this DOM
    went_dark = delta[(current.reindex(delta.index, fill_value=0) <= 0) & (baseline.reindex(delta.index, fill_value=0) > 0)]
    went_dark = went_dark.sort_values()
    return {
        "delta": delta,
        "decliners": decliners,
        "gainers": gainers,
        "went_dark": went_dark,
    }


def mtd_avg_pace_curve(daily_dom: pd.DataFrame) -> pd.Series:
    """Global MTD average pace (kbd) at each day-of-month in the frame."""
    if daily_dom.empty:
        return pd.Series(dtype=float)
    daily_total = daily_dom.sum(axis=1)
    return pd.Series(
        {d: daily_total.loc[:d].mean() for d in daily_total.index},
        dtype=float,
    )


def build_pace_history(product: str, month: int, years: list[int]) -> pd.DataFrame:
    """day-of-month x year matrix of global MTD average pace (kbd)."""
    df = fetch_origin_countries_flows(product)
    cols = {}
    for y in years:
        dom = slice_month_by_country(df, y, month)
        if not dom.empty:
            cols[y] = mtd_avg_pace_curve(dom)
    return pd.DataFrame(cols).sort_index()


def build_country_hist_pace(
    product: str, month: int, years: list[int], thru_dom: int
) -> pd.Series:
    """Per-country median MTD-avg pace at thru_dom across prior same-calendar-months."""
    df = fetch_origin_countries_flows(product)
    by_year: list[pd.Series] = []
    for y in years:
        dom = slice_month_by_country(df, y, month)
        if not dom.empty:
            by_year.append(country_mean_pace(dom, thru_dom))
    if not by_year:
        return pd.Series(dtype=float)
    return pd.DataFrame(by_year).median(axis=0)


def dom_cumulative_global(daily_dom: pd.DataFrame) -> pd.Series:
    """Cumsum of global daily kbd (Σ countries) — Section 4-style cumulative MTD curve."""
    if daily_dom.empty:
        return pd.Series(dtype=float)
    return daily_dom.sum(axis=1).sort_index().cumsum()


def build_cum_history_matrix(product: str, month: int, years: list[int]) -> pd.DataFrame:
    """day-of-month × year matrix of global cumulative MTD exports."""
    df = fetch_origin_countries_flows(product)
    cols = {}
    for y in years:
        dom = slice_month_by_country(df, y, month)
        if not dom.empty:
            cols[y] = dom_cumulative_global(dom)
    return pd.DataFrame(cols).sort_index()


def analyze_product_pace(product: str) -> dict:
    df = fetch_origin_countries_flows(product)

    cur_dom = slice_month_by_country(df, PACE_YEAR, PACE_MONTH, as_of=_pace_as_of)
    prior_dom = slice_month_by_country(df, _py_prior, _pm_prior)
    ly_dom = slice_month_by_country(df, PACE_YEAR - 1, PACE_MONTH)

    cur_pace = country_mean_pace(cur_dom, _pace_dom)
    prior_pace = country_mean_pace(prior_dom, _pace_dom)
    ly_pace = country_mean_pace(ly_dom, _pace_dom)

    hist_years = list(range(PACE_YEAR - PACE_N_HISTORY, PACE_YEAR))
    hist_country_pace = build_country_hist_pace(
        product, PACE_MONTH, hist_years, _pace_dom
    )

    vs_prior = pace_contributors(cur_pace, prior_pace, top_n=PACE_TOP_N)
    vs_ly = pace_contributors(cur_pace, ly_pace, top_n=PACE_TOP_N)
    vs_hist = pace_contributors(cur_pace, hist_country_pace, top_n=PACE_TOP_N)

    global_cur = global_mean_pace(cur_dom, _pace_dom)
    global_prior = global_mean_pace(prior_dom, _pace_dom)
    global_ly = global_mean_pace(ly_dom, _pace_dom)

    hist = build_pace_history(product, PACE_MONTH, hist_years)
    band = seasonal_band(hist) if not hist.empty else pd.DataFrame()

    median_at_dom = float(band["median"].get(_pace_dom, float("nan"))) if not band.empty else float("nan")
    p25_at_dom = float(band["p25"].get(_pace_dom, float("nan"))) if not band.empty else float("nan")
    p75_at_dom = float(band["p75"].get(_pace_dom, float("nan"))) if not band.empty else float("nan")

    pct_rank = None
    if not hist.empty and _pace_dom in hist.index and global_cur == global_cur:
        row = hist.loc[_pace_dom].dropna()
        if not row.empty:
            pct_rank = float((row < global_cur).mean() * 100)

    vs_hist_median_kbd = global_cur - median_at_dom if global_cur == global_cur else float("nan")

    # Inflection heuristics — flag supply-relevant shifts for daily monitoring
    inflection_flags = []
    if global_cur == global_cur and p25_at_dom == p25_at_dom and global_cur < p25_at_dom:
        inflection_flags.append("below_5y_p25")
    if global_cur == global_cur and global_prior == global_prior:
        chg = global_cur - global_prior
        if global_prior > 0 and chg / global_prior < -0.10:
            inflection_flags.append("pace_down_10pct_vs_prior_mo")
        if chg > 0 and global_prior > 0 and chg / global_prior > 0.10:
            inflection_flags.append("pace_up_10pct_vs_prior_mo")
    if global_cur == global_cur and median_at_dom == median_at_dom:
        chg = global_cur - median_at_dom
        if median_at_dom > 0 and chg / median_at_dom < -0.10:
            inflection_flags.append("pace_down_10pct_vs_5y_med")
        if chg > 0 and median_at_dom > 0 and chg / median_at_dom > 0.10:
            inflection_flags.append("pace_up_10pct_vs_5y_med")

    cur_curve = mtd_avg_pace_curve(cur_dom)
    prior_curve = mtd_avg_pace_curve(prior_dom)
    ly_curve = mtd_avg_pace_curve(ly_dom)

    hist_cum = build_cum_history_matrix(product, PACE_MONTH, hist_years)
    cum_band = seasonal_band(hist_cum) if not hist_cum.empty else pd.DataFrame()
    cur_cum = dom_cumulative_global(cur_dom)
    prior_cum = dom_cumulative_global(prior_dom)
    ly_cum = dom_cumulative_global(ly_dom)

    return {
        "product": product,
        "cur_pace": cur_pace,
        "global_pace_kbd": global_cur,
        "vs_prior_mo_kbd": global_cur - global_prior,
        "vs_last_yr_kbd": global_cur - global_ly,
        "vs_hist_median_kbd": vs_hist_median_kbd,
        "median_kbd": median_at_dom,
        "p25_kbd": p25_at_dom,
        "p75_kbd": p75_at_dom,
        "percentile": pct_rank,
        "inflection_flags": inflection_flags,
        "vs_prior": vs_prior,
        "vs_ly": vs_ly,
        "vs_hist": vs_hist,
        "hist_country_pace": hist_country_pace,
        "cur_curve": cur_curve,
        "prior_curve": prior_curve,
        "ly_curve": ly_curve,
        "band": band,
        "hist": hist,
        "cur_cum": cur_cum,
        "prior_cum": prior_cum,
        "ly_cum": ly_cum,
        "cum_band": cum_band,
        "hist_cum": hist_cum,
    }


# Baseline keys inside each analyze_product_pace() result — no extra API calls for heatmaps.
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PACE_DELTA_BASELINES = {
    "prior_mo": {
        "contrib_key": "vs_prior",
        "title": f"vs {calendar.month_name[_pm_prior]} {_py_prior}",
        "colorbar": "Δpace vs prior mo (kbd)",
    },
    "last_yr": {
        "contrib_key": "vs_ly",
        "title": f"vs {_pace_month_name} {PACE_YEAR - 1}",
        "colorbar": "Δpace vs last year (kbd)",
    },
    "hist_median": {
        "contrib_key": "vs_hist",
        "title": f"vs {PACE_N_HISTORY}y {_pace_month_name} median",
        "colorbar": f"Δpace vs {PACE_N_HISTORY}y med (kbd)",
    },
}


def select_top_countries_for_heatmap(
    results: dict,
    *,
    baseline_keys: tuple[str, ...] = ("vs_prior", "vs_ly", "vs_hist"),
    top_n: int = 18,
) -> pd.Index:
    """Rank countries by largest |Δpace| across products and baselines (pure pandas)."""
    chunks: list[pd.Series] = []
    for contrib_key in baseline_keys:
        for product_result in results.values():
            chunks.append(product_result[contrib_key]["delta"].abs())
    if not chunks:
        return pd.Index([])
    score = pd.concat(chunks, axis=1).fillna(0).max(axis=1)
    return score.nlargest(top_n).index


def build_pace_delta_matrix(
    results: dict,
    *,
    baseline: str,
    top_n: int = 18,
    top_countries: pd.Index | None = None,
) -> pd.DataFrame:
    """
    Countries × products matrix of Δpace (kbd) for one baseline.

    Uses precomputed PACE_RESULTS only — zero additional Kpler API calls.
    """
    spec = PACE_DELTA_BASELINES[baseline]
    contrib_key = spec["contrib_key"]
    matrix = pd.DataFrame(
        {product: r[contrib_key]["delta"] for product, r in results.items()}
    )
    if top_countries is None:
        top_countries = select_top_countries_for_heatmap(results, top_n=top_n)
    matrix = matrix.reindex(top_countries).fillna(0)
    matrix.index = [short_country(c) for c in matrix.index]
    return matrix


def _format_delta_text(values) -> list[list[str]]:
    """Pre-format heatmap cell labels (avoids long float strings in Plotly)."""
    return [
        [f"{v:+.0f}" if v == v else "" for v in row]
        for row in values
    ]


def make_global_pace_panel(
    results: dict,
    products: list[str],
    *,
    mode: str = "avg",
    show: bool = True,
) -> go.Figure:
    """
    Multi-product pace monitor — Section 4 style.

    mode='avg'  → MTD average export rate (kbd)
    mode='cum'  → cumulative MTD exports (Σ daily kbd, kb-equivalent)
    """
    if mode == "cum":
        band_key, prior_key, ly_key, cur_key = "cum_band", "prior_cum", "ly_cum", "cur_cum"
        y_label = "cumulative kb (MTD)"
        title = f"Global cumulative export pace — {_pace_month_name} {PACE_YEAR} (DOM {_pace_dom})"
        subplot_suffix = "cumulative MTD"
    else:
        band_key, prior_key, ly_key, cur_key = "band", "prior_curve", "ly_curve", "cur_curve"
        y_label = "kbd (MTD avg)"
        title = f"Global export pace monitor — {_pace_month_name} {PACE_YEAR} (DOM {_pace_dom})"
        subplot_suffix = "kbd MTD avg"

    if USE_MOCK:
        title += "  [MOCK]"

    prior_label = f"{calendar.month_name[_pm_prior]} {_py_prior}"
    ly_label = f"{_pace_month_name} {PACE_YEAR - 1}"
    cur_label = f"{_pace_month_name} {PACE_YEAR}"

    fig = make_subplots(
        rows=len(products),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.07,
        subplot_titles=[f"{p} — global export pace ({subplot_suffix})" for p in products],
    )

    for i, product in enumerate(products, start=1):
        r = results[product]
        band = r[band_key]
        x = band.index if not band.empty else r[cur_key].index

        if not band.empty:
            fig.add_trace(
                go.Scatter(x=x, y=band["max"], line=dict(width=0), hoverinfo="skip", showlegend=False),
                row=i, col=1,
            )
            fig.add_trace(
                go.Scatter(
                    x=x, y=band["min"], fill="tonexty", line=dict(width=0),
                    fillcolor="rgba(158,202,225,0.30)", name="5y range (min–max)", hoverinfo="skip",
                    legendgroup="band", showlegend=(i == 1),
                ),
                row=i, col=1,
            )
            fig.add_trace(
                go.Scatter(x=x, y=band["p75"], line=dict(width=0), hoverinfo="skip", showlegend=False),
                row=i, col=1,
            )
            fig.add_trace(
                go.Scatter(
                    x=x, y=band["p25"], fill="tonexty", line=dict(width=0),
                    fillcolor="rgba(107,174,214,0.45)", name="5y IQR (25–75%)", hoverinfo="skip",
                    legendgroup="iqr", showlegend=(i == 1),
                ),
                row=i, col=1,
            )
            fig.add_trace(
                go.Scatter(
                    x=x, y=band["median"], name="5y median",
                    line=dict(color="#2171b5", width=2),
                    legendgroup="med", showlegend=(i == 1),
                ),
                row=i, col=1,
            )

        fig.add_trace(
            go.Scatter(
                x=r[prior_key].index, y=r[prior_key].values, name=prior_label,
                line=dict(color="#9e9ac8", width=2),
                legendgroup="prior", showlegend=(i == 1),
            ),
            row=i, col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=r[ly_key].index, y=r[ly_key].values, name=ly_label,
                line=dict(color="#74c476", width=2),
                legendgroup="ly", showlegend=(i == 1),
            ),
            row=i, col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=r[cur_key].index, y=r[cur_key].values, name=cur_label,
                line=dict(color="#e6550d", width=3, dash="dot"),
                legendgroup="cur", showlegend=(i == 1),
            ),
            row=i, col=1,
        )
        fig.add_vline(x=_pace_dom, line_width=1, line_dash="dash", line_color="#888", row=i, col=1)

        flags = None#r["inflection_flags"]
        if flags and not r[cur_key].empty:
            fig.add_annotation(
                x=1,
                y=float(r[cur_key].max()),
                text="⚑ " + ", ".join(flags),
                showarrow=False,
                font=dict(size=10, color="#b45309"),
                xanchor="right",
                row=i,
                col=1,
            )

    fig.update_layout(
        height=280 * len(products),
        template="plotly_white",
        title=title,
        legend=dict(orientation="h", y=-0.02),
        margin=dict(t=80, b=60),
    )
    fig.update_xaxes(title_text="Day of month", dtick=2, range=[1, 31], row=len(products), col=1)
    fig.update_yaxes(title_text=y_label, col=1)

    if show:
        fig.show()
    return fig


def _contributor_bar_series(contrib: dict, top_n: int) -> pd.Series:
    """Top decliners + gainers for a diverging horizontal bar chart."""
    delta = contrib["delta"]
    neg = delta[delta < 0].nsmallest(top_n)
    pos = delta[delta > 0].nlargest(top_n)
    return pd.concat([neg, pos]).sort_values()


def make_contributor_bar_panel(
    results: dict,
    products: list[str],
    *,
    top_n: int = 5,
    row_height: int = 220,
    show: bool = True,
) -> go.Figure:
    """Product × baseline grid of horizontal contributor bar charts."""
    bar_baselines = [
        (f"{calendar.month_name[_pm_prior]} {_py_prior}", "vs_prior", "vs_prior_mo_kbd"),
        (f"{_pace_month_name} {PACE_YEAR - 1}", "vs_ly", "vs_last_yr_kbd"),
        (f"{PACE_N_HISTORY}y med", "vs_hist", "vs_hist_median_kbd"),
    ]
    n_rows, n_cols = len(products), len(bar_baselines)

    # Per-cell titles (denser than row/column title gutters); global Δ in title not on bars
    subplot_titles = []
    for product in products:
        r = results[product]
        for bl_label, _, global_key in bar_baselines:
            g = r[global_key]
            gtxt = f"{g:+,.0f}" if g == g else "—"
            subplot_titles.append(f"{product} · {bl_label}  (Δ {gtxt})")

    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.10,
        vertical_spacing=0.08,
        specs=[[{"type": "bar"}] * n_cols for _ in products],
    )

    for row_i, product in enumerate(products, start=1):
        r = results[product]
        for col_i, (_, contrib_key, _) in enumerate(bar_baselines, start=1):
            series = _contributor_bar_series(r[contrib_key], top_n)
            if series.empty:
                continue
            colors = ["#c0392b" if v < 0 else "#27ae60" for v in series]
            fig.add_trace(
                go.Bar(
                    y=[short_country(c) for c in series.index],
                    x=series.values,
                    orientation="h",
                    marker_color=colors,
                    showlegend=False,
                    hovertemplate="%{y}<br>Δpace: %{x:+,.0f} kbd<extra></extra>",
                ),
                row=row_i,
                col=col_i,
            )
            fig.add_vline(x=0, line_width=1, line_color="#ccc", row=row_i, col=col_i)
            fig.update_yaxes(autorange="reversed", automargin=True, row=row_i, col=col_i)
            fig.update_xaxes(tickformat=",", automargin=True, row=row_i, col=col_i)

    for col_i in range(1, n_cols + 1):
        fig.update_xaxes(title_text="Δpace (kbd)", title_standoff=8, row=n_rows, col=col_i)

    fig.update_layout(
        height=row_height * n_rows + 80,
        template="plotly_white",
        title=(
            f"Country contributors to pace change — {_pace_month_name} {PACE_YEAR} (DOM {_pace_dom})"
            + ("  [MOCK]" if USE_MOCK else "")
        ),
        margin=dict(t=70, b=55, l=85, r=25),
        bargap=0.15,
    )
    fig.update_annotations(font_size=10)

    if show:
        fig.show()
    return fig


def make_pace_delta_heatmap(
    results: dict,
    *,
    baseline: str = "hist_median",
    top_n: int = 18,
    top_countries: pd.Index | None = None,
    zmax: float | None = None,
) -> tuple[pd.DataFrame, go.Figure]:
    """Single baseline heatmap figure + underlying data matrix."""
    spec = PACE_DELTA_BASELINES[baseline]
    matrix = build_pace_delta_matrix(
        results,
        baseline=baseline,
        top_n=top_n,
        top_countries=top_countries,
    )
    if matrix.empty:
        return matrix, go.Figure()

    _zmax = zmax
    if _zmax is None:
        _zmax = max(float(matrix.abs().max().max()), 1.0)

    fig = go.Figure(
        go.Heatmap(
            z=matrix.values,
            x=list(matrix.columns),
            y=list(matrix.index),
            zmin=-_zmax,
            zmax=_zmax,
            colorscale="RdYlGn",
            zmid=0,
            colorbar=dict(title=spec["colorbar"]),
            hovertemplate=(
                "Country: %{y}<br>Product: %{x}<br>Δpace: %{z:+,.0f} kbd<extra></extra>"
            ),
            text=_format_delta_text(matrix.values),
            texttemplate="%{text}",
            textfont={"size": 9},
        )
    )
    fig.update_layout(
        title=(
            f"{spec['title']} — {_pace_month_name} {PACE_YEAR} (DOM {_pace_dom})"
            + ("  [MOCK]" if USE_MOCK else "")
        ),
        template="plotly_white",
        height=max(420, 24 * len(matrix) + 140),
        xaxis=dict(side="top"),
        margin=dict(l=130, r=60, t=80, b=40),
    )
    return matrix, fig


def make_pace_delta_heatmap_panel(
    results: dict,
    *,
    baselines: tuple[str, ...] = ("prior_mo", "last_yr", "hist_median"),
    top_n: int = 18,
    show: bool = True,
) -> tuple[dict[str, pd.DataFrame], go.Figure]:
    """
    Three-panel country×product heatmap (prior mo / last yr / 5y median).

    One pass over PACE_RESULTS — no API calls. Same country set on all panels
    so you can compare how a mover looks across baselines.
    """
    top_countries = select_top_countries_for_heatmap(results, top_n=top_n)
    matrices: dict[str, pd.DataFrame] = {}
    zmax = 1.0
    for baseline in baselines:
        matrices[baseline] = build_pace_delta_matrix(
            results,
            baseline=baseline,
            top_countries=top_countries,
        )
        zmax = max(zmax, float(matrices[baseline].abs().max().max()))

    fig = make_subplots(
        rows=len(baselines),
        cols=1,
        row_titles=[PACE_DELTA_BASELINES[b]["title"] for b in baselines],
        vertical_spacing=0.06,
        specs=[[{"type": "heatmap"}] for _ in baselines],
    )

    for row_i, baseline in enumerate(baselines, start=1):
        matrix = matrices[baseline]
        fig.add_trace(
            go.Heatmap(
                z=matrix.values,
                x=list(matrix.columns),
                y=list(matrix.index),
                zmin=-zmax,
                zmax=zmax,
                colorscale="RdYlGn",
                zmid=0,
                showscale=(row_i == len(baselines)),
                colorbar=dict(title="Δpace (kbd)", len=0.28, y=0.15) if row_i == len(baselines) else None,
                hovertemplate=(
                    "Country: %{y}<br>Product: %{x}<br>Δpace: %{z:+,.0f} kbd<extra></extra>"
                ),
                text=_format_delta_text(matrix.values),
                texttemplate="%{text}",
                textfont={"size": 9},
            ),
            row=row_i,
            col=1,
        )
        fig.update_xaxes(side="top", tickangle=-30, row=row_i, col=1)
        fig.update_yaxes(automargin=True, row=row_i, col=1)

    panel_h = max(260, 20 * len(top_countries))
    fig.update_layout(
        title=(
            f"Country export pace changes — {_pace_month_name} {PACE_YEAR} (DOM {_pace_dom})"
            + ("  [MOCK]" if USE_MOCK else "")
        ),
        template="plotly_white",
        height=panel_h * len(baselines) + 140,
        margin=dict(l=130, r=70, t=90, b=40),
    )
    fig.update_annotations(font_size=12, xanchor="right", x=1.0)

    if show:
        fig.show()

    return matrices, fig


print(
    f"Fetching global OriginCountries flows for {len(PACE_PRODUCTS)} products "
    f"({_pace_month_name} {PACE_YEAR}, DOM {_pace_dom})…"
)
PACE_RESULTS = {p: analyze_product_pace(p) for p in PACE_PRODUCTS}
print("Done.")

Fetching global OriginCountries flows for 4 products (October 2026, DOM 16)…
Done.


In [9]:
def _fmt_kbd(x: float) -> str:
    if x != x:
        return "—"
    sign = "+" if x > 0 else ""
    return f"{sign}{x:,.0f} kbd"


def _fmt_row(r: dict) -> dict:
    pct = r["percentile"]
    return {
        "product": r["product"],
        "pace_kbd": round(r["global_pace_kbd"], 1) if r["global_pace_kbd"] == r["global_pace_kbd"] else None,
        "vs_prior_mo": round(r["vs_prior_mo_kbd"], 1) if r["vs_prior_mo_kbd"] == r["vs_prior_mo_kbd"] else None,
        "vs_last_yr": round(r["vs_last_yr_kbd"], 1) if r["vs_last_yr_kbd"] == r["vs_last_yr_kbd"] else None,
        "vs_5y_med": round(r["vs_hist_median_kbd"], 1) if r["vs_hist_median_kbd"] == r["vs_hist_median_kbd"] else None,
        "5y_median": round(r["median_kbd"], 1) if r["median_kbd"] == r["median_kbd"] else None,
        "pctile": f"{pct:.0f}th" if pct is not None else "—",
        "flags": ", ".join(r["inflection_flags"]) or "—",
    }


pace_summary = pd.DataFrame(_fmt_row(r) for r in PACE_RESULTS.values()).set_index("product")
display(
    pace_summary.style.format(
        {
            "pace_kbd": "{:,.1f}",
            "vs_prior_mo": "{:+,.1f}",
            "vs_last_yr": "{:+,.1f}",
            "vs_5y_med": "{:+,.1f}",
            "5y_median": "{:,.1f}",
        }
    ).set_caption(
        f"Global export pace — {_pace_month_name} {PACE_YEAR} (DOM {_pace_dom}, kbd MTD avg)"
        + ("  [MOCK]" if USE_MOCK else "")
    )
)


def format_contributor_block(
    product: str,
    contrib: dict,
    *,
    baseline_label: str,
    global_delta: float | None = None,
) -> str:
    """Desk-style contributor note — headline follows sign of global pace change."""
    if global_delta is not None and global_delta == global_delta:
        direction = "decline" if global_delta < 0 else "increase"
    else:
        direction = "decline"
    lines = [f"{product}", ""]
    headline = {
        "decline": f"Main contributors to the decline in export pace in {_pace_month_name} (vs {baseline_label}):",
        "increase": f"Main contributors to the increase in export pace in {_pace_month_name} (vs {baseline_label}):",
    }[direction]
    lines.append(headline)
    lines.append("")

    decliners = contrib["decliners"]
    gainers = contrib["gainers"]
    went_dark = contrib["went_dark"]

    if not decliners.empty:
        for country, dlt in decliners.items():
            note = " (No cargos exported so far)" if country in went_dark.index else ""
            lines.append(f"{short_country(country)} {_fmt_kbd(dlt)}{note}")
        lines.append("")

    if not gainers.empty:
        for country, dlt in gainers.items():
            lines.append(f"{short_country(country)} {_fmt_kbd(dlt)}")
        lines.append("")

    extra_dark = went_dark.drop(decliners.index, errors="ignore")
    if not extra_dark.empty:
        names = ", ".join(short_country(c) for c in extra_dark.index[:12])
        suffix = ", etc" if len(extra_dark) > 12 else ""
        lines.append(f"Also no exports so far (had pace in {baseline_label}): {names}{suffix}")
        lines.append("")

    return "\n".join(lines)


def print_all_contributor_notes():
    print("=" * 72)
    print(f"Deep dive on product pace contributors — {_pace_month_name} {PACE_YEAR} (DOM {_pace_dom})")
    print("=" * 72)
    for product in PACE_PRODUCTS:
        r = PACE_RESULTS[product]
        print()
        print(format_contributor_block(
            product,
            r["vs_prior"],
            baseline_label=f"{calendar.month_name[_pm_prior]} {_py_prior}",
            global_delta=r["vs_prior_mo_kbd"],
        ))
        print("— vs last year —")
        print(format_contributor_block(
            product,
            r["vs_ly"],
            baseline_label=f"{_pace_month_name} {PACE_YEAR - 1}",
            global_delta=r["vs_last_yr_kbd"],
        ))
        print(f"— vs {PACE_N_HISTORY}y {_pace_month_name} median (DOM {_pace_dom}) —")
        print(format_contributor_block(
            product,
            r["vs_hist"],
            baseline_label=f"{PACE_N_HISTORY}y {_pace_month_name} median",
            global_delta=r["vs_hist_median_kbd"],
        ))


print_all_contributor_notes()

,pace_kbd,vs_prior_mo,vs_last_yr,vs_5y_med,5y_median,pctile,flags
product,,,,,,,
Gasoline,"3,918.0",-407.2,-694.5,-677.2,"4,595.2",0th,"below_5y_p25, pace_down_10pct_vs_5y_med"
Jet,"1,317.3",-171.5,-312.5,-312.5,"1,629.8",20th,"below_5y_p25, pace_down_10pct_vs_prior_mo, pace_down_10pct_vs_5y_med"
Gasoil/Diesel,"5,089.7","-1,367.4","-1,547.3","-1,767.9","6,857.5",0th,"below_5y_p25, pace_down_10pct_vs_prior_mo, pace_down_10pct_vs_5y_med"
FO,"2,356.3",-296.2,"-1,053.8","-1,078.2","3,434.5",0th,"below_5y_p25, pace_down_10pct_vs_prior_mo, pace_down_10pct_vs_5y_med"


Deep dive on product pace contributors — October 2026 (DOM 16)

Gasoline

Main contributors to the decline in export pace in October (vs September 2026):

USA -79 kbd
France -76 kbd
Italy -75 kbd
Saudi Arabia -58 kbd
Cyprus -54 kbd

Netherlands +115 kbd
UK +57 kbd
China +47 kbd
Portugal +40 kbd
Singapore Republic +27 kbd

Also no exports so far (had pace in September 2026): Argentina, Democratic Republic of the Congo, Ireland, Bulgaria, Poland

— vs last year —
Gasoline

Main contributors to the decline in export pace in October (vs October 2025):

Saudi Arabia -227 kbd
Singapore Republic -217 kbd
UAE -152 kbd
China -103 kbd
Italy -74 kbd

Netherlands +158 kbd
Belgium +80 kbd
Sweden +59 kbd
Norway +57 kbd
Spain +49 kbd

Also no exports so far (had pace in October 2025): Togo, Algeria, Australia, Qatar, Croatia, Egypt, Kuwait, Vietnam, Peru, Ireland, Panama, Argentina

— vs 5y October median (DOM 16) —
Gasoline

Main contributors to the decline in export pace in October (vs 5y October m

In [10]:
def country_comparison_table(r: dict, *, top_n: int = 12) -> pd.DataFrame:
    """Top countries ranked by |Δ vs 5y median| with all three baseline deltas."""
    d_hist = r["vs_hist"]["delta"]
    top = d_hist.abs().nlargest(top_n).index
    cur = r["cur_pace"]
    return pd.DataFrame(
        {
            "pace_kbd": cur.reindex(top, fill_value=0),
            "vs_prior_mo": r["vs_prior"]["delta"].reindex(top, fill_value=0),
            "vs_last_yr": r["vs_ly"]["delta"].reindex(top, fill_value=0),
            "vs_5y_med": d_hist.reindex(top, fill_value=0),
            f"{PACE_N_HISTORY}y_med": r["hist_country_pace"].reindex(top, fill_value=0),
        },
        index=[short_country(c) for c in top],
    )


# --- Styled tables: one per product (top movers vs historical pattern) ---
for product in PACE_PRODUCTS:
    tbl = country_comparison_table(PACE_RESULTS[product])
    _vmax = max(float(tbl["vs_5y_med"].abs().max()), 1.0)
    display(
        tbl.style.format("{:+,.0f}", subset=["vs_prior_mo", "vs_last_yr", "vs_5y_med"])
        .format("{:,.0f}", subset=["pace_kbd", f"{PACE_N_HISTORY}y_med"])
        .background_gradient(
            cmap="RdYlGn", subset=["vs_5y_med"], vmin=-_vmax, vmax=_vmax
        )
        .set_caption(
            f"{product} — top countries by |Δ vs {PACE_N_HISTORY}y median| "
            f"({_pace_month_name} {PACE_YEAR}, DOM {_pace_dom})"
        )
    )

# --- Contributor bar charts + heatmaps (functions defined above; zero extra API calls) ---
PACE_CONTRIB_FIG = make_contributor_bar_panel(PACE_RESULTS, PACE_PRODUCTS, top_n=PACE_TOP_N)
PACE_HEATMAP_DATA, PACE_HEATMAP_FIG = make_pace_delta_heatmap_panel(PACE_RESULTS, top_n=18)

,pace_kbd,vs_prior_mo,vs_last_yr,vs_5y_med,5y_med
Saudi Arabia,9,-58,-227,-227,235
UAE,nan,+nan,+nan,+nan,nan
Netherlands,525,+115,+158,+158,367
Belgium,154,+8,+80,-122,276
Singapore Republic,394,+27,-217,-108,502
China,152,+47,-103,-103,255
India,243,-18,-45,+103,140
Italy,65,-75,-74,-82,147
Nigeria,78,-49,+38,+78,0
USA,nan,+nan,+nan,+nan,nan


,pace_kbd,vs_prior_mo,vs_last_yr,vs_5y_med,5y_med
USA,nan,+nan,+nan,+nan,nan
China,91,-173,-160,-152,243
Kuwait,35,-24,-111,-136,171
Nigeria,96,-40,+26,+96,0
UAE,nan,+nan,+nan,+nan,nan
South Korea,306,+123,+123,+74,232
Saudi Arabia,146,+87,+52,+57,90
Unknown,44,-66,+44,+44,0
Singapore Republic,21,-38,-56,-30,50
Egypt,0,+0,-29,-29,29


,pace_kbd,vs_prior_mo,vs_last_yr,vs_5y_med,5y_med
Russia,nan,+nan,+nan,+nan,nan
Saudi Arabia,255,-246,-388,-388,643
USA,nan,+nan,+nan,+nan,nan
UAE,nan,+nan,+nan,+nan,nan
Kuwait,32,-110,-268,-218,250
India,722,+176,+74,+202,521
Singapore Republic,156,-71,+13,-135,291
Malaysia,120,-13,-27,-123,243
Japan,32,-88,-101,-101,133
Unknown,96,-208,+96,+96,0


,pace_kbd,vs_prior_mo,vs_last_yr,vs_5y_med,5y_med
Iran,84,+29,-146,-322,406
UAE,nan,+nan,+nan,+nan,nan
Unknown,0,-43,-112,-112,112
Netherlands,57,-21,-43,-107,165
Kuwait,117,+54,+6,+98,19
Singapore Republic,127,-43,-73,-73,200
USA,nan,+nan,+nan,+nan,nan
Malaysia,251,+8,-51,-63,314
Saudi Arabia,0,-161,-87,-58,58
Brazil,93,-170,-77,-55,148


In [11]:
# MTD average export rate (kbd) — run-rate view
PACE_AVG_PACE_FIG = make_global_pace_panel(PACE_RESULTS, PACE_PRODUCTS, mode="avg")

### 7b. Cumulative export pace (Section 4 style)

Same global products as above, but plotted as **cumulative MTD exports** (Σ daily kbd by day-of-month) —
matching the chart in **Section 4**. Useful for reading total volume shipped vs prior month, last year,
and the 5-year seasonal band (not just the run-rate).

In [12]:
# Cumulative MTD curve — Section 4 convention (no extra API calls)
PACE_CUM_PACE_FIG = make_global_pace_panel(PACE_RESULTS, PACE_PRODUCTS, mode="cum")

### 7c. Monthly volume comparison (kb)

Full-month export totals from the cumulative curves above — levels plus m/m and y/y changes in **kb** and **%**.
Use the MMbbl row (kb ÷ 1,000) when cross-checking Kpler UI monthly totals.

In [13]:
def _month_total_kb(cum: pd.Series) -> float:
    """Last point on the cumulative MTD curve = total volume shipped in the month (kb)."""
    if cum is None or cum.empty:
        return float("nan")
    return float(cum.iloc[-1])


def _pct_chg(cur: float, base: float) -> float:
    if cur != cur or base != base or base == 0:
        return float("nan")
    return (cur - base) / base * 100


def build_volume_comparison_table(results: dict, products: list[str]) -> pd.DataFrame:
    cur_label = f"{_pace_month_name} {PACE_YEAR}"
    prior_label = f"{calendar.month_name[_pm_prior]} {_py_prior}"
    ly_label = f"{_pace_month_name} {PACE_YEAR - 1}"

    rows = []
    for product in products:
        r = results[product]
        cur = _month_total_kb(r["cur_cum"])
        prior = _month_total_kb(r["prior_cum"])
        ly = _month_total_kb(r["ly_cum"])
        rows.append(
            {
                "product": product,
                f"{cur_label} (kb)": cur,
                f"{prior_label} (kb)": prior,
                f"{ly_label} (kb)": ly,
                "Δ m/m (kb)": cur - prior if cur == cur and prior == prior else float("nan"),
                "Δ m/m (%)": _pct_chg(cur, prior),
                "Δ y/y (kb)": cur - ly if cur == cur and ly == ly else float("nan"),
                "Δ y/y (%)": _pct_chg(cur, ly),
            }
        )
    return pd.DataFrame(rows).set_index("product")


vol_compare = build_volume_comparison_table(PACE_RESULTS, PACE_PRODUCTS)
level_cols = [c for c in vol_compare.columns if c.endswith("(kb)") and not c.startswith("Δ")]

display(
    vol_compare.style.format(
        {
            **{c: "{:,.0f}" for c in level_cols},
            "Δ m/m (kb)": "{:+,.0f}",
            "Δ m/m (%)": "{:+.1f}",
            "Δ y/y (kb)": "{:+,.0f}",
            "Δ y/y (%)": "{:+.1f}",
        },
        na_rep="—",
    ).set_caption(
        f"Global export volume — OriginCountries, realized only, intra-country={PACE_WITH_INTRA_COUNTRY}"
        + ("  [MOCK]" if USE_MOCK else "")
    )
)

mmbbl = vol_compare[level_cols].div(1000)
mmbbl.columns = [c.replace("(kb)", "(MMbbl)") for c in mmbbl.columns]
display(
    mmbbl.style.format("{:,.2f}", na_rep="—").set_caption(
        "Level columns in MMbbl (kb ÷ 1,000) — handy for Kpler UI monthly totals"
    )
)

,October 2026 (kb),September 2026 (kb),October 2025 (kb),Δ m/m (kb),Δ m/m (%),Δ y/y (kb),Δ y/y (%)
product,,,,,,,
Gasoline,"62,688","130,787","147,301","-68,099",-52.1,"-84,612",-57.4
Jet,"21,077","50,538","52,457","-29,461",-58.3,"-31,380",-59.8
Gasoil/Diesel,"81,435","190,202","201,712","-108,767",-57.2,"-120,277",-59.6
FO,"37,700","78,930","103,850","-41,230",-52.2,"-66,150",-63.7


,October 2026 (MMbbl),September 2026 (MMbbl),October 2025 (MMbbl)
product,,,
Gasoline,62.69,130.79,147.30
Jet,21.08,50.54,52.46
Gasoil/Diesel,81.43,190.20,201.71
FO,37.70,78.93,103.85


In [14]:
from kpler.sdk import Platform
from kpler.sdk.configuration import Configuration
from kpler.sdk.resources.flows import Flows

config = Configuration(Platform.Liquids, KPLER_EMAIL, KPLER_PASSWORD)

from datetime import date
from kpler.sdk.resources.flows import Flows
from kpler.sdk import FlowsDirection, FlowsSplit, FlowsPeriod, FlowsMeasurementUnit
flows_client = Flows(config)
flows_client.get(
    flow_direction=[FlowsDirection.Export],
    split=[FlowsSplit.OriginCountries],
    granularity=[FlowsPeriod.Daily],
    unit=[FlowsMeasurementUnit.T],
    start_date=date(2020,9,1),
    end_date=date(2020,10,1)
)

,Date,Saudi Arabia,Russian Federation,United States,United Arab Emirates,Iraq,Kuwait,Nigeria,Brazil,South Korea,...,Moldova,Honduras,Suriname,Iceland,Haiti,Djibouti,Kenya,Maldives,Faroe Islands,Period End Date
0,2020-09-01,1337934.21,636090.02,574187.94,841001.94,399524.13,457877.08,24237.74,124774.25,264182.31,...,0.00,0.00,0.00,0.00,4400.16,0.00,0.00,0.00,0.00,2020-09-01
1,2020-09-02,958901.00,1058528.42,236515.66,652074.27,0.00,341756.27,749881.02,79240.48,66681.61,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2020-09-02
2,2020-09-03,532283.41,298858.00,892535.97,511625.21,854799.39,376110.10,302693.16,381104.26,0.00,...,0.00,0.00,2358.72,0.00,0.00,0.00,0.00,0.00,0.00,2020-09-03
3,2020-09-04,832319.81,761950.98,718690.58,267976.37,288216.71,258271.38,49758.94,46458.20,317234.83,...,0.00,0.00,3313.44,0.00,0.00,0.00,0.00,0.00,0.00,2020-09-04
4,2020-09-05,1044677.22,738837.72,770725.54,400307.04,423682.49,76125.00,125941.60,466515.17,244945.26,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2020-09-05
5,2020-09-06,1442576.81,998301.42,265866.33,337562.39,508708.32,207326.63,245257.20,221406.29,111709.76,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2020-09-06
6,2020-09-07,768620.68,741639.03,571420.83,281749.56,392340.60,510043.93,136662.41,65928.09,2294.76,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2020-09-07
7,2020-09-08,790715.18,950712.94,879194.52,382281.83,684521.38,511456.62,360994.82,470171.26,46550.17,...,0.00,0.00,0.00,5696.15,0.00,0.00,0.00,0.00,0.00,2020-09-08
8,2020-09-09,1087583.24,553295.77,614260.97,1025767.76,315924.45,23067.85,384081.82,81642.88,129193.76,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2020-09-09
9,2020-09-10,777694.46,849886.47,377071.79,538033.88,496473.45,321808.18,392159.34,176169.69,306681.22,...,5385.93,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2020-09-10


In [15]:
from kpler.sdk import (FlowsDirection, FlowsSplit, FlowsPeriod,
                        FlowsMeasurementUnit)
unit_enum = {"kb": FlowsMeasurementUnit.KB,
                "kbd": FlowsMeasurementUnit.KBD}[UNIT]

live = False

In [16]:
flows_client.get(
        flow_direction=[FlowsDirection.Export],
        split=[FlowsSplit.Total],
        granularity=[FlowsPeriod.Daily],
        from_zones=LOCATIONS,
        products=PRODUCTS,
        unit=[unit_enum],
        start_date=date(2019, 1, 1),
        end_date=date.today(),
        with_forecast=WITH_FORECAST_LIVE if live else WITH_FORECAST_HIST,
        only_realized=False if live else ONLY_REALIZED_HIST,
        with_intra_country=WITH_INTRA_COUNTRY,
    )

,Date,Total,Period End Date
0,2019-01-01,1239.77,2019-01-01
1,2019-01-02,340.56,2019-01-02
2,2019-01-03,1515.38,2019-01-03
3,2019-01-04,503.27,2019-01-04
4,2019-01-05,789.12,2019-01-05
...,...,...,...
2841,2026-10-12,274.15,2026-10-12
2842,2026-10-13,298.45,2026-10-13
2843,2026-10-14,0.00,2026-10-14
2844,2026-10-15,0.00,2026-10-15


In [17]:
# 1) Confirm entity types
from kpler.sdk.resources.zones import Zones
zones_client = Zones(config)
for name in ["Yanbu", "Fujairah", "Oman"]:
    print(name, "->", zones_client.search(name))

# 2) Compare totals: zones-only vs split buckets vs possible overlap
#    e.g. Oman alone + Ras Tanura alone vs Oman + Ras Tanura together

Yanbu ->               zones
0             Yanbu
1  Yanbu Commercial
2      Yanbu Light.
Fujairah ->                 zones
0            Fujairah
1     Fujairah Light.
2  Fujairah Anchorage
3   Fujairah F2 Plant
Oman ->           zones
0          Oman
1  Gulf of Oman


In [18]:
flows_client.get(
        flow_direction=[FlowsDirection.Export],
        split=[FlowsSplit.Total],
        granularity=[FlowsPeriod.Daily],
        from_zones='Oman',
        products=PRODUCTS,
        unit=[unit_enum],
        start_date=date(2026, 1, 1),
        end_date=date.today(),
        with_forecast=WITH_FORECAST_LIVE if live else WITH_FORECAST_HIST,
        only_realized=False if live else ONLY_REALIZED_HIST,
        with_intra_country=WITH_INTRA_COUNTRY,
    )

,Date,Total,Period End Date
0,2026-01-01,318.67,2026-01-01
1,2026-01-02,0.00,2026-01-02
2,2026-01-03,0.00,2026-01-03
3,2026-01-04,0.00,2026-01-04
4,2026-01-05,255.35,2026-01-05
...,...,...,...
284,2026-10-12,0.00,2026-10-12
285,2026-10-13,0.00,2026-10-13
286,2026-10-14,0.00,2026-10-14
287,2026-10-15,0.00,2026-10-15


In [19]:
flows_client.get(
        flow_direction=[FlowsDirection.Export],
        split=[FlowsSplit.Total],
        granularity=[FlowsPeriod.Daily],
        from_zones='Oman',
        #from_installations = 'Ras Tanura',
        products=PRODUCTS,
        unit=[unit_enum],
        start_date=date(2026, 1, 1),
        end_date=date.today(),
        with_forecast=WITH_FORECAST_LIVE if live else WITH_FORECAST_HIST,
        only_realized=False if live else ONLY_REALIZED_HIST,
        with_intra_country=WITH_INTRA_COUNTRY,
    )

,Date,Total,Period End Date
0,2026-01-01,318.67,2026-01-01
1,2026-01-02,0.00,2026-01-02
2,2026-01-03,0.00,2026-01-03
3,2026-01-04,0.00,2026-01-04
4,2026-01-05,255.35,2026-01-05
...,...,...,...
284,2026-10-12,0.00,2026-10-12
285,2026-10-13,0.00,2026-10-13
286,2026-10-14,0.00,2026-10-14
287,2026-10-15,0.00,2026-10-15


In [20]:
zones_client.search("Ras Tanura")      # probably returns a hit


,zones
0,Ras Tanura


In [21]:
zones_client.search("Duqm Refinery")   # probably empty or different name

,zones


## 8. Monthly export table — location × product explorer

Pull **monthly export totals** from Kpler Flows (`granularity=Monthly`, `split=Total`) and display
as a wide table: **rows = months**, **columns = products**.

**Filters**
- **Zones** (`from_zones`) — country, region, port, or custom trading zone (e.g. `"China"`)
- **Installations** (`from_installations`) — specific origin terminal/refinery
- Kpler rejects zone + installation in one request, so both are fetched separately and summed
- **Products** — one API call per product (Flows returns a single combined `Total` otherwise)
- **Unit** — `kb` (default), `kbd`, or `kt` / `kilotonnes`

**Two outputs** (separate functions):
1. `show_monthly_exports_table()` — raw monthly values
2. `show_monthly_exports_yoy_table()` — YoY % change per product vs same month last year

In [22]:
from IPython.display import display

# ---------------------- Section 8 — monthly export table ----------------------
# Re-run this cell after edits (kernel keeps old function definitions otherwise).

_MONTHLY_CACHE: dict = {}


def _monthly_unit_enum(unit: str) -> tuple[str, object, float]:
    """Normalize unit string, map to Kpler enum, return display scale."""
    from kpler.sdk import FlowsMeasurementUnit

    aliases = {
        "kb": "kb",
        "kbd": "kbd",
        "kt": "kt",
        "kilotonne": "kt",
        "kilotonnes": "kt",
    }
    key = aliases.get(unit.strip().lower())
    if key is None:
        raise ValueError(
            f"Unknown unit {unit!r}; use kb, kbd, kt/kilotonnes"
        )

    # Prefer native KT; fall back to T (tonnes) and scale to kt if needed.
    kt_enum = getattr(FlowsMeasurementUnit, "KT", None)
    if key == "kt" and kt_enum is not None:
        return key, kt_enum, 1.0
    if key == "kt":
        return key, FlowsMeasurementUnit.T, 0.001

    mapping = {
        "kb": FlowsMeasurementUnit.KB,
        "kbd": FlowsMeasurementUnit.KBD,
    }
    return key, mapping[key], 1.0


def _monthly_location_label(
    zones: str | list[str] | None,
    installations: str | list[str] | None,
) -> str:
    parts = as_filter_list(zones) + as_filter_list(installations)
    return filter_label(parts) if parts else "—"


def _merge_monthly_frames(frames: list[pd.DataFrame]) -> pd.Series:
    """Sum monthly Total series when zone + installation need separate API calls."""
    series_list: list[pd.Series] = []
    for df in frames:
        if df.empty or "Date" not in df.columns:
            continue
        dt = pd.to_datetime(df["Date"])
        if "Total" in df.columns:
            vals = pd.to_numeric(df["Total"], errors="coerce")
        else:
            vals = df.select_dtypes("number").sum(axis=1)
        series_list.append(pd.Series(vals.to_numpy(), index=dt))
    if not series_list:
        return pd.Series(dtype=float)
    merged = pd.concat(series_list, axis=1).fillna(0).sum(axis=1)
    merged.index = pd.to_datetime(merged.index)
    return merged.sort_index()


def _mock_monthly_exports(
    product: str,
    zones: list[str],
    installations: list[str],
    unit_key: str,
    start_date: date,
    end_date: date,
) -> pd.Series:
    """Synthetic monthly totals when USE_MOCK=True."""
    idx = pd.date_range(start_date, end_date, freq="MS")
    seed = hash(
        (tuple(zones), tuple(installations), product, unit_key)
    ) % (2**31)
    rng = np.random.default_rng(seed)
    scale = {"kb": 8000.0, "kbd": 280.0, "kt": 1.0}[unit_key]
    season = 1.0 + 0.1 * np.sin((idx.month - 1) / 12 * 2 * np.pi)
    trend = 1.0 + 0.02 * (idx.year - start_date.year)
    noise = rng.uniform(0.7, 1.3, len(idx))
    return pd.Series(scale * season * trend * noise, index=idx, name=product)


def fetch_monthly_exports(
    *,
    zones: str | list[str] | None = None,
    installations: str | list[str] | None = None,
    products: str | list[str],
    unit: str = "kb",
    start_date: date | None = None,
    end_date: date | None = None,
    with_forecast: bool = False,
    only_realized: bool = True,
    with_intra_country: bool = True,
) -> pd.DataFrame:
    """
    Monthly export totals: rows = YYYY-MM, columns = products.

    Pass zones and/or installations; Kpler needs separate calls per location
    filter type and per product.
    """
    zones = as_filter_list(zones)
    installations = as_filter_list(installations)
    if not zones and not installations:
        raise ValueError("Pass at least one zone or installation")
    product_list = as_filter_list(products)
    if not product_list:
        raise ValueError("Pass at least one product")

    unit_key, unit_enum, unit_scale = _monthly_unit_enum(unit)
    start_date = start_date or date(2020, 1, 1)
    end_date = end_date or date.today()

    key = (
        tuple(zones),
        tuple(installations),
        tuple(product_list),
        unit_key,
        start_date,
        end_date,
        with_forecast,
        only_realized,
        with_intra_country,
        USE_MOCK,
    )
    if key in _MONTHLY_CACHE:
        return _MONTHLY_CACHE[key].copy()

    columns: dict[str, pd.Series] = {}
    for product in product_list:
        if USE_MOCK:
            columns[product] = _mock_monthly_exports(
                product, zones, installations, unit_key, start_date, end_date
            )
            continue

        from kpler.sdk import FlowsDirection, FlowsPeriod, FlowsSplit

        loc_calls: list[dict] = []
        if zones:
            loc_calls.append({"from_zones": zones})
        if installations:
            loc_calls.append({"from_installations": installations})

        frames = []
        for loc_kw in loc_calls:
            df = _get_flows_client().get(
                flow_direction=[FlowsDirection.Export],
                split=[FlowsSplit.Total],
                granularity=[FlowsPeriod.Monthly],
                products=[product],
                unit=[unit_enum],
                start_date=start_date,
                end_date=end_date,
                with_forecast=with_forecast,
                only_realized=only_realized,
                with_intra_country=with_intra_country,
                **loc_kw,
            )
            frames.append(df)
        columns[product] = _merge_monthly_frames(frames) * unit_scale

    out = pd.DataFrame(columns).sort_index()
    out.index = pd.to_datetime(out.index).to_period("M").astype(str)
    out.index.name = "month"
    _MONTHLY_CACHE[key] = out.copy()
    return out


def monthly_exports_yoy_pct(raw: pd.DataFrame) -> pd.DataFrame:
    """YoY % change per product: (month / same month last year - 1) * 100."""
    if raw.empty:
        return raw.copy()
    periods = pd.PeriodIndex(raw.index, freq="M")
    out = pd.DataFrame(index=raw.index, columns=raw.columns, dtype=float)
    for col in raw.columns:
        cur = pd.Series(raw[col].values, index=periods)
        prior = cur.shift(12)
        out[col] = ((cur / prior) - 1) * 100
    out.index.name = "month"
    return out


def show_monthly_exports_table(
    *,
    zones: str | list[str] | None = None,
    installations: str | list[str] | None = None,
    products: str | list[str],
    unit: str = "kb",
    start_date: date | None = None,
    end_date: date | None = None,
    with_forecast: bool = False,
    only_realized: bool = True,
    with_intra_country: bool = True,
) -> pd.DataFrame:
    """Fetch and display raw monthly export totals."""
    raw = fetch_monthly_exports(
        zones=zones,
        installations=installations,
        products=products,
        unit=unit,
        start_date=start_date,
        end_date=end_date,
        with_forecast=with_forecast,
        only_realized=only_realized,
        with_intra_country=with_intra_country,
    )
    unit_key, _, _ = _monthly_unit_enum(unit)
    caption = (
        f"Monthly exports ({unit_key}) — "
        f"{_monthly_location_label(zones, installations)} — "
        f"{filter_label(products)}"
        + ("  (MOCK DATA)" if USE_MOCK else "")
    )
    display(
        raw.style.format("{:,.0f}", na_rep="—").set_caption(caption)
    )
    return raw


def show_monthly_exports_yoy_table(
    raw: pd.DataFrame | None = None,
    *,
    zones: str | list[str] | None = None,
    installations: str | list[str] | None = None,
    products: str | list[str] | None = None,
    unit: str = "kb",
    start_date: date | None = None,
    end_date: date | None = None,
    with_forecast: bool = False,
    only_realized: bool = True,
    with_intra_country: bool = True,
) -> pd.DataFrame:
    """Display YoY % change table; pass `raw` to avoid a second API pull."""
    if raw is None:
        if products is None:
            raise ValueError("Pass `raw` or the same fetch kwargs (incl. products)")
        raw = fetch_monthly_exports(
            zones=zones,
            installations=installations,
            products=products,
            unit=unit,
            start_date=start_date,
            end_date=end_date,
            with_forecast=with_forecast,
            only_realized=only_realized,
            with_intra_country=with_intra_country,
        )
    yoy = monthly_exports_yoy_pct(raw)
    caption = (
        f"YoY % change — "
        f"{_monthly_location_label(zones, installations)} — "
        f"{', '.join(raw.columns)}"
        + ("  (MOCK DATA)" if USE_MOCK else "")
    )
    display(
        yoy.style.format("{:+.1f}%", na_rep="—")
        .background_gradient(cmap="RdYlGn", axis=None, vmin=-50, vmax=50)
        .set_caption(caption)
    )
    return yoy

In [23]:
# ---------------------- China product exports — raw monthly table ----------------------
# Run the Section 8 functions cell immediately above first.
_monthly_unit_enum("kt")  # quick sanity check — raises if that cell is stale

MONTHLY_ZONES = "China"
MONTHLY_INSTALLATIONS = None#"Zhejiang Zhoushan Refinery and Petrochemical"   # e.g. "Some terminal" — use zones_client.search() to confirm name
MONTHLY_PRODUCTS = ["Gasoline", "Jet", "Gasoil/Diesel", "FO"]
MONTHLY_UNIT = "kt"            # also: "kb", "kbd"
MONTHLY_START = date(2026, 1, 1)
MONTHLY_END = date.today()
MONTHLY_WITH_FORECAST = False
MONTHLY_ONLY_REALIZED = True
MONTHLY_WITH_INTRA_COUNTRY = False

china_monthly = show_monthly_exports_table(
    zones=MONTHLY_ZONES,
    installations=MONTHLY_INSTALLATIONS,
    products=MONTHLY_PRODUCTS,
    unit=MONTHLY_UNIT,
    start_date=MONTHLY_START,
    end_date=MONTHLY_END,
    with_forecast=MONTHLY_WITH_FORECAST,
    only_realized=MONTHLY_ONLY_REALIZED,
    with_intra_country=MONTHLY_WITH_INTRA_COUNTRY,
)

,Gasoline,Jet,Gasoil/Diesel,FO
month,,,,
2026-01,385,"1,006",413,0
2026-02,260,"1,143",870,45
2026-03,345,970,637,0
2026-04,53,539,153,168
2026-05,88,695,192,9
2026-06,114,663,315,218
2026-07,688,683,729,0
2026-08,938,962,"1,245",79
2026-09,583,"1,017","1,856",164


In [24]:
# YoY % change per product (reuses china_monthly — no extra API calls)
china_yoy = show_monthly_exports_yoy_table(
    raw=china_monthly,
    zones=MONTHLY_ZONES,
    installations=MONTHLY_INSTALLATIONS,
)

,Gasoline,Jet,Gasoil/Diesel,FO
month,,,,
2026-01,—,—,—,—
2026-02,—,—,—,—
2026-03,—,—,—,—
2026-04,—,—,—,—
2026-05,—,—,—,—
2026-06,—,—,—,—
2026-07,—,—,—,—
2026-08,—,—,—,—
2026-09,—,—,—,—
